# **PII Middleware**
Detect and handle Personally Identifiable Information (PII) in conversations. This middleware detects common PII types and applies configurable strategies to handle them. It can detect emails, credit cards, IP addresses, MAC addresses, and URLs in both user input and agent output.

**Configuration options:**
- pii_type: `Literal['email', 'credit_card', 'ip', 'mac_address', 'url']` | `str`
- strategy:
    - block: Raise an exception when PII is detected
    - redact: Replace PII with `[REDACTED_TYPE]` placeholders
    - mask: Partially mask PII (e.g., `****-****-****-1234` for credit card)
    - hash: Replace PII with deterministic hash (e.g., `<email_hash:a1b2c3d4>`)
- apply_to_input: bool = True
- apply_to_output: bool = False
- apply_to_tool_results: bool = False

In [11]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=chat_model,
    tools=[],
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True, apply_to_output=True),
    ],
)

response = agent.invoke(
    {
        "messages": "generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number"
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number
================================== Ai Message ==================================

I can generate a Markdown table with 5 random datapoints, but keep in mind that generating real credit card numbers is difficult due to the strict regulations around their use and format. I will use a placeholder in the format "XXXX-XXXX-XXXX-XXXX" for the credit card number, as actual credit card data is sensitive and should never be shared lightly.

### Random Datapoints Table
| Name        | Email                | IP Address       | Credit Card   | Country  |
|-------------|----------------------|------------------|----------------|----------|
| Emily Chen  | [REDACTED_EMAIL]    | 192.168.1.100    | 1234-5678-9012-3456  | USA      |
| Ethan Lee   | [REDACTED_EMAIL]     | 10.0.0.1         | 5678-9012-3456-